# GUTS on Cora

The Full-Graph baseline, GUTS source runs and shard unlearning of node and edge forget sets on Cora (K=20),
for four models over 10 seeds (2026-2035), summarized as mean ± standard deviation.

## Settings

Paths, device, seeds, models, forget sets and run directories.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "guts").is_dir() and (REPO_ROOT / "configs").is_dir(), \
    "Start Jupyter from the repository root."
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from guts.method.utils import timestamp_id
from guts.static_graph import cora_data, forget_sets, full_graph, pipeline, unlearn

In [2]:
DEVICE = "cuda:0"                         # one device; the runs go one after another
SEEDS = list(range(2026, 2036))           # run seed (split, partition, training and forget set)
MODELS = ["graphsage", "gcn", "gat", "gin"]
CONFIG = "configs/cora.yaml"              # K=20 and the other settings of the paper

SPLIT_DIR = "processed/Cora_split"        # stratified 80/20 split per seed

FORGET_TYPES = ["node", "edge"]
FORGET_PERCENT = 0.5                      # % of the train nodes / of the edges between train nodes
FORGET_DIR = "processed/forget_sets/Cora"

BASE_ROOT = REPO_ROOT / "Results/runs/Cora"
SHARD_ROOT = REPO_ROOT / "Results/shard-runs/Cora"
UNLEARN_ROOT = REPO_ROOT / "Results/shard-unlearns/Cora"
RUN_ID = timestamp_id()                   # new runs each time this cell runs


def split_path(seed):
    return f"{SPLIT_DIR}/cora_seed{seed}.pt"


def forget_path(forget_type, seed):
    return f"{FORGET_DIR}/{forget_type.capitalize()}/forget_{forget_type}_seed{seed}_{FORGET_PERCENT:g}pct.json"


def base_dir(model, seed):
    return BASE_ROOT / f"{RUN_ID}_full_graph_{model}_seed{seed}"


def shard_dir(model, seed):
    return SHARD_ROOT / f"{RUN_ID}_guts_{model}_seed{seed}"


def unlearn_dir(model, seed, forget_type):
    return UNLEARN_ROOT / f"{RUN_ID}_guts_{model}_seed{seed}_{forget_type}_{FORGET_PERCENT:g}pct"


for run_dir in (base_dir("<model>", "<seed>"), shard_dir("<model>", "<seed>"),
                unlearn_dir("<model>", "<seed>", "<forget type>")):
    print("run directory:", run_dir.relative_to(REPO_ROOT))

run directory: Results/runs/Cora/261024_03h32m01s_full_graph_<model>_seed<seed>
run directory: Results/shard-runs/Cora/261024_03h32m01s_guts_<model>_seed<seed>
run directory: Results/shard-unlearns/Cora/261024_03h32m01s_guts_<model>_seed<seed>_<forget type>_0.5pct


## Data

The split file of every seed and the node and edge forget sets drawn from its train nodes.

In [3]:
for seed in SEEDS:
    if (REPO_ROOT / split_path(seed)).exists():
        print("split exists:", split_path(seed))
    else:
        cora_data.main(["split", "--seed", str(seed), "--out", split_path(seed)])

split exists: processed/Cora_split/cora_seed2026.pt
split exists: processed/Cora_split/cora_seed2027.pt
split exists: processed/Cora_split/cora_seed2028.pt
split exists: processed/Cora_split/cora_seed2029.pt
split exists: processed/Cora_split/cora_seed2030.pt
split exists: processed/Cora_split/cora_seed2031.pt
split exists: processed/Cora_split/cora_seed2032.pt
split exists: processed/Cora_split/cora_seed2033.pt
split exists: processed/Cora_split/cora_seed2034.pt
split exists: processed/Cora_split/cora_seed2035.pt


In [4]:
for seed in SEEDS:
    missing = []
    for forget_type in FORGET_TYPES:
        if (REPO_ROOT / forget_path(forget_type, seed)).exists():
            print("forget set exists:", forget_path(forget_type, seed))
        else:
            missing.append(forget_type)
    if missing:
        forget_sets.main(["--data_path", split_path(seed), "--percent", f"{FORGET_PERCENT:g}",
                          "--types", *missing, "--seed", str(seed), "--out_root", FORGET_DIR])

forget set exists: processed/forget_sets/Cora/Node/forget_node_seed2026_0.5pct.json
forget set exists: processed/forget_sets/Cora/Edge/forget_edge_seed2026_0.5pct.json
forget set exists: processed/forget_sets/Cora/Node/forget_node_seed2027_0.5pct.json
forget set exists: processed/forget_sets/Cora/Edge/forget_edge_seed2027_0.5pct.json
forget set exists: processed/forget_sets/Cora/Node/forget_node_seed2028_0.5pct.json
forget set exists: processed/forget_sets/Cora/Edge/forget_edge_seed2028_0.5pct.json
forget set exists: processed/forget_sets/Cora/Node/forget_node_seed2029_0.5pct.json
forget set exists: processed/forget_sets/Cora/Edge/forget_edge_seed2029_0.5pct.json
forget set exists: processed/forget_sets/Cora/Node/forget_node_seed2030_0.5pct.json
forget set exists: processed/forget_sets/Cora/Edge/forget_edge_seed2030_0.5pct.json
forget set exists: processed/forget_sets/Cora/Node/forget_node_seed2031_0.5pct.json
forget set exists: processed/forget_sets/Cora/Edge/forget_edge_seed2031_0.5p

## Full-Graph baseline

One model trained on the whole train subgraph and evaluated on the full graph.

In [5]:
for model in MODELS:
    for seed in SEEDS:
        run_dir = base_dir(model, seed)
        if (run_dir / "results.json").exists():
            print("finished run exists:", run_dir.relative_to(REPO_ROOT))
            continue
        full_graph.main(["--config", CONFIG, "--data_path", split_path(seed), "--model_type", model,
                         "--seed", str(seed), "--device", DEVICE,
                         "--out_root", str(run_dir.parent), "--run_name", run_dir.name])

[03:33:25] Full-Graph baseline (static graph)  2026-10-24  run=Results/runs/Cora/261024_03h32m01s_full_graph_graphsage_seed2026  seed=2026 (given)  model=graphsage


[03:33:25]   data=processed/Cora_split/cora_seed2026.pt  nodes 2708  edges 10556  classes 7  train 2166  test 542
[03:33:25]   G_train  nodes 2166  edges 6464
[03:33:25] STEP 1/2 training (graphsage, 100 epochs, final-epoch model)
[03:33:27]   training done  train nodes 2166  1.1s
[03:33:27] STEP 2/2 evaluation (test, G_full)
[03:33:27]   test  ACC 0.8745  F1 0.8594  (542 test nodes, 1 model)  0.0s
[03:33:27] done  2s  -> Results/runs/Cora/261024_03h32m01s_full_graph_graphsage_seed2026/results.json
[03:33:27] Full-Graph baseline (static graph)  2026-10-24  run=Results/runs/Cora/261024_03h32m01s_full_graph_graphsage_seed2027  seed=2027 (given)  model=graphsage
[03:33:27]   data=processed/Cora_split/cora_seed2027.pt  nodes 2708  edges 10556  classes 7  train 2166  test 542
[03:33:27]   G_train  nodes 2166  edges 6910
[03:33:27] STEP 1/2 training (graphsage, 100 epochs, final-epoch model)
[03:33:28]   training done  train nodes 2166  0.5s
[03:33:28] STEP 2/2 evaluation (test, G_full)
[03:

## GUTS source runs

GUTS partitioning of the train subgraph, one model per shard, and the aggregated evaluation on the full graph.

In [6]:
for model in MODELS:
    for seed in SEEDS:
        run_dir = shard_dir(model, seed)
        if (run_dir / "results.json").exists():
            print("finished run exists:", run_dir.relative_to(REPO_ROOT))
            continue
        pipeline.main(["--config", CONFIG, "--data_path", split_path(seed), "--model_type", model,
                       "--seed", str(seed), "--device", DEVICE,
                       "--override", f"output.root={run_dir.parent}", "--run_name", run_dir.name])

[03:34:09] GUTS static-graph pipeline  2026-10-24  run=Results/shard-runs/Cora/261024_03h32m01s_guts_graphsage_seed2026  seed=2026 (given)  model=graphsage  K=20
[03:34:09]   data=processed/Cora_split/cora_seed2026.pt  nodes 2708  edges 10556  classes 7  train 2166  test 542
[03:34:09]   G_train  nodes 2166  edges 6464
[03:34:09] STEP 1/4 partition (K=20)
[03:34:11]   partition done  balance 0.1622  fairness 0.0558  2.3s
[03:34:11] STEP 2/4 shard subgraphs
[03:34:11]   20 shards built  nodes 59-206
[03:34:11] STEP 3/4 shard training (graphsage, 100 epochs, final-epoch models)
[03:34:21]   20/20 shards trained  10s
[03:34:21] STEP 4/4 aggregated evaluation (test, G_full, mean softmax over the shard models)
[03:34:21]   test  ACC 0.8081  F1 0.7728  (542 test nodes, 20 models)  0.0s
[03:34:21] done  12s  -> Results/shard-runs/Cora/261024_03h32m01s_guts_graphsage_seed2026/results.json
[03:34:21] GUTS static-graph pipeline  2026-10-24  run=Results/shard-runs/Cora/261024_03h32m01s_guts_graph

## Unlearning

Each forget set removed from its GUTS run: only the affected shards are rebuilt and retrained, and the run is
evaluated on the clean graph.

In [7]:
for model in MODELS:
    for seed in SEEDS:
        if not (shard_dir(model, seed) / "results.json").exists():
            raise FileNotFoundError(f"{shard_dir(model, seed).relative_to(REPO_ROOT)} not finished: "
                                    "run the GUTS source cell first")
        for forget_type in FORGET_TYPES:
            run_dir = unlearn_dir(model, seed, forget_type)
            if (run_dir / "results_unlearn.json").exists():
                print("finished run exists:", run_dir.relative_to(REPO_ROOT))
                continue
            unlearn.main(["--run_dir", str(shard_dir(model, seed)), "--forget_set", forget_path(forget_type, seed),
                          "--device", DEVICE, "--out_root", str(run_dir.parent), "--run_name", run_dir.name])

[03:44:04] GUTS static-graph unlearn (node)  2026-10-24  source=Results/shard-runs/Cora/261024_03h32m01s_guts_graphsage_seed2026  out=Results/shard-unlearns/Cora/261024_03h32m01s_guts_graphsage_seed2026_node_0.5pct  model=graphsage
[03:44:04]   forget set processed/forget_sets/Cora/Node/forget_node_seed2026_0.5pct.json  11 targets
[03:44:04]   clean graph built in memory (deleted=11 nodes, train 2166->2155, edges 10556->10460)
[03:44:04]   affected shards 7/20
[03:44:07]   7/7 affected shards retrained  3s
[03:44:07]   test  ACC 0.8229  F1 0.7920  (542 test nodes, 20 models)  0.0s
[03:44:07] done  3s  -> Results/shard-unlearns/Cora/261024_03h32m01s_guts_graphsage_seed2026_node_0.5pct/results_unlearn.json
[03:44:07] GUTS static-graph unlearn (edge)  2026-10-24  source=Results/shard-runs/Cora/261024_03h32m01s_guts_graphsage_seed2026  out=Results/shard-unlearns/Cora/261024_03h32m01s_guts_graphsage_seed2026_edge_0.5pct  model=graphsage
[03:44:07]   forget set processed/forget_sets/Cora/Edg

## Results

Prediction metrics of the runs on the test nodes, the partition quality of the GUTS runs and the affected shards
of the unlearning runs (mean ± sample standard deviation over the seeds).

In [8]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def mean_pm_std(values, digits=4):
    return f"{values.mean():.{digits}f} ± {values.std():.{digits}f}"  # pandas std: sample (ddof=1)


rows = []
for model in MODELS:
    for seed in SEEDS:
        base = load_json(base_dir(model, seed) / "results.json")
        shard = load_json(shard_dir(model, seed) / "results.json")
        rows.append({"model": model, "method": "Full-Graph", "seed": seed,
                     "ACC": base["test_acc"], "F1": base["test_f1_macro"]})
        rows.append({"model": model, "method": f"GUTS (K={shard['num_shards']})", "seed": seed,
                     "ACC": shard["test_acc"], "F1": shard["test_f1_macro"],
                     "balance": shard["partition_balance"], "fairness": shard["partition_fairness"]})
        for forget_type in FORGET_TYPES:
            unlearn_res = load_json(unlearn_dir(model, seed, forget_type) / "results_unlearn.json")
            rows.append({"model": model, "method": f"GUTS unlearn ({forget_type} {FORGET_PERCENT:g}%)",
                         "seed": seed, "ACC": unlearn_res["test_acc"], "F1": unlearn_res["test_f1_macro"],
                         "affected shards": unlearn_res["num_affected"]})
per_run = pd.DataFrame(rows)
K = load_json(shard_dir(MODELS[0], SEEDS[0]) / "results.json")["num_shards"]

by_method = per_run.groupby(["model", "method"], sort=False)
prediction = by_method[["ACC", "F1"]].agg(mean_pm_std)
prediction["runs"] = by_method.size()
display(prediction)

ACC               F1  runs
model     method                                                          
graphsage Full-Graph                0.8655 ± 0.0135  0.8541 ± 0.0124    10
          GUTS (K=20)               0.8096 ± 0.0116  0.7800 ± 0.0198    10
          GUTS unlearn (node 0.5%)  0.8055 ± 0.0133  0.7760 ± 0.0159    10
          GUTS unlearn (edge 0.5%)  0.8052 ± 0.0138  0.7767 ± 0.0171    10
gcn       Full-Graph                0.8753 ± 0.0089  0.8640 ± 0.0102    10
          GUTS (K=20)               0.8127 ± 0.0146  0.7834 ± 0.0172    10
          GUTS unlearn (node 0.5%)  0.8153 ± 0.0168  0.7843 ± 0.0187    10
          GUTS unlearn (edge 0.5%)  0.8096 ± 0.0132  0.7794 ± 0.0158    10
gat       Full-Graph                0.8923 ± 0.0086  0.8828 ± 0.0091    10
          GUTS (K=20)               0.7498 ± 0.0172  0.7009 ± 0.0323    10
          GUTS unlearn (node 0.5%)  0.7509 ± 0.0200  0.7029 ± 0.0319    10
          GUTS unlearn (edge 0.5%)  0.7517 ± 0.0156  0.7030 ± 0.0278    10
gin       Full-Graph                0.8579 ± 0.0123  0.8486 ± 0.0128    10
          GUTS (K=20)               0.8197 ± 0.0127  0.7960 ± 0.0154    10
          GUTS unlearn (node 0.5%)  0.8192 ± 0.0098  0.7946 ± 0.0150    10
          GUTS unlearn (edge 0.5%)  0.8162 ± 0.0138  0.7919 ± 0.0151    10